In [9]:
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

connection = psycopg2.connect(
    host=os.getenv("PGHOST", "localhost"),
    port=int(os.getenv("PGPORT", "5432")),
    database=os.getenv("PGDATABASE", "postgres"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "password")
)

In [10]:
#pip install psycopg2-binary

In [11]:
import psycopg2
from dateutil import parser

# Your JSON-like data
data = [
    {'start_date': 'Jan 1st, 2025', 'end_date': 'Dec 31st, 2025', 'project_manager_name': 'Sabin Dcruz',
     'project_id': 'GEHC OTR Analytics 2024', 'total_cost': '$42,000', 'project_no': 'CO#1', 'currency_code': '$'},
    {'start_date': '01-Jan-2025', 'end_date': '31-Dec-2025', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02046', 'total_cost': '$98,112.00', 'project_no': '8', 'currency_code': 'USD'},
    {'start_date': '01-Jan-2025', 'end_date': '31-Dec-2025', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02045', 'total_cost': '147,168.00', 'project_no': '8', 'currency_code': 'USD'},
    {'start_date': '01-Feb-2025', 'end_date': '31-Jan-2026', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02062', 'total_cost': '$128,400.00', 'project_no': '3', 'currency_code': 'USD'},
    {'start_date': '01-Feb-2025', 'end_date': '31-Jan-2026', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02290', 'total_cost': '$36,000.00', 'project_no': '3', 'currency_code': 'USD'},
    {'start_date': '01-Oct-2023', 'end_date': '31-Dec-2025', 'project_manager_name': 'Lisa Prestegaard',
     'project_id': 'PSBC02078', 'total_cost': '$788,256', 'project_no': 'PSBC01411', 'currency_code': '$'},
    {'start_date': '2024-01-01', 'end_date': '2025-12-31', 'project_manager_name': 'Noah Benbazza',
     'project_id': 'PSBC02100', 'total_cost': '100,200.00', 'project_no': '1', 'currency_code': 'USD'},
    {'start_date': 'Jan 1st, 2025', 'end_date': 'Dec 31st, 2026', 'project_manager_name': 'S. Jacob Nesa Raj',
     'project_id': 'PSBC02138', 'total_cost': '$434,202', 'project_no': 'PSBC02138', 'currency_code': '$'},
    {'start_date': '01st February 2025', 'end_date': 'January 31, 2026', 'project_manager_name': 'Joseph DeLuca',
     'project_id': 'PSBC02188', 'total_cost': '$44,040', 'project_no': 'GEHC Payment Tracking and S2P Support 2025-26', 'currency_code': '$'},
    {'start_date': 'Mar 01, 2025', 'end_date': 'Feb 28, 2026', 'project_manager_name': 'Scott Velasquez',
     'project_id': 'PSBC02203', 'total_cost': '$70,000', 'project_no': '6', 'currency_code': '$'},
    {'start_date': '3rd-February-2025', 'end_date': '31st-January-2026', 'project_manager_name': 'Ramneet Saluja',
     'project_id': 'GEHC CRM Integration Interface Support', 'total_cost': '145,152', 'project_no': None, 'currency_code': 'USD'}
]

# DB Connection
connection = psycopg2.connect(
    host="localhost",
    port="5432",
    database="postgres",
    user="postgres",
    password="password"
)
cursor = connection.cursor()

# Drop and recreate table with correct schema
cursor.execute("DROP TABLE IF EXISTS Coupa_contracts CASCADE;")
cursor.execute("""
CREATE TABLE Coupa_contracts (
    project_manager_name VARCHAR(100),
    project_id VARCHAR(100) PRIMARY KEY,
    project_no VARCHAR(50),
    start_date DATE,
    end_date DATE,
    total_cost NUMERIC,
    currency_code VARCHAR(10)
);
""")
connection.commit()
print("✅ Table recreated with total_cost column")

# Insert query
insert_query = """
INSERT INTO Coupa_contracts 
(project_manager_name, project_id, project_no, start_date, end_date, total_cost, currency_code)
VALUES (%s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (project_id) DO NOTHING;
"""

# Insert data
for record in data:
    try:
        start_date = parser.parse(record["start_date"]).date()
        end_date = parser.parse(record["end_date"]).date()
    except Exception as e:
        print(f"❌ Date parse failed for {record['project_id']}: {e}")
        continue

    # Clean cost: remove $ and , and convert to float
    cost_str = str(record.get("total_cost", "0")).replace("$", "").replace(",", "").strip()
    try:
        total_cost = float(cost_str)
    except ValueError:
        total_cost = None

    values = (
        record["project_manager_name"],
        record["project_id"],
        record.get("project_no"),
        start_date,
        end_date,
        total_cost,
        record.get("currency_code")
    )
    cursor.execute(insert_query, values)

connection.commit()
print(f"✅ {len(data)} records inserted")

# Fetch and print inserted data
cursor.execute("SELECT * FROM Coupa_contracts ORDER BY project_id;")
for row in cursor.fetchall():
    print(row)

cursor.close()
connection.close()

✅ Table recreated with total_cost column
✅ 11 records inserted
('Ramneet Saluja', 'GEHC CRM Integration Interface Support', None, datetime.date(2025, 2, 3), datetime.date(2026, 1, 31), Decimal('145152.0'), 'USD')
('Sabin Dcruz', 'GEHC OTR Analytics 2024', 'CO#1', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('42000.0'), '$')
('Rama Singh', 'PSBC02045', '8', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('147168.0'), 'USD')
('Rama Singh', 'PSBC02046', '8', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('98112.0'), 'USD')
('Rama Singh', 'PSBC02062', '3', datetime.date(2025, 2, 1), datetime.date(2026, 1, 31), Decimal('128400.0'), 'USD')
('Lisa Prestegaard', 'PSBC02078', 'PSBC01411', datetime.date(2023, 10, 1), datetime.date(2025, 12, 31), Decimal('788256.0'), '$')
('Noah Benbazza', 'PSBC02100', '1', datetime.date(2024, 1, 1), datetime.date(2025, 12, 31), Decimal('100200.0'), 'USD')
('S. Jacob Nesa Raj', 'PSBC02138', 'PSBC02138', datetime

In [12]:
import psycopg2
from dateutil import parser

# Your JSON-like data
data = [
    {'start_date': 'Jan 1st, 2025', 'end_date': 'Dec 31st, 2025', 'project_manager_name': 'Sabin Dcruz',
     'project_id': 'GEHC OTR Analytics 2024', 'total_cost': '$42,000', 'project_no': 'CO#1', 'currency_code': '$'},
    {'start_date': '01-Jan-2025', 'end_date': '31-Dec-2025', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02046', 'total_cost': '$98,112.00', 'project_no': '8', 'currency_code': 'USD'},
    {'start_date': '01-Jan-2025', 'end_date': '31-Dec-2025', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02045', 'total_cost': '147,168.00', 'project_no': '8', 'currency_code': 'USD'},
    {'start_date': '01-Feb-2025', 'end_date': '31-Jan-2026', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02062', 'total_cost': '$128,400.00', 'project_no': '3', 'currency_code': 'USD'},
    {'start_date': '01-Feb-2025', 'end_date': '31-Jan-2026', 'project_manager_name': 'Rama Singh',
     'project_id': 'PSBC02290', 'total_cost': '$36,000.00', 'project_no': '3', 'currency_code': 'USD'},
    {'start_date': '01-Oct-2023', 'end_date': '31-Dec-2025', 'project_manager_name': 'Lisa Prestegaard',
     'project_id': 'PSBC02078', 'total_cost': '$788,256', 'project_no': 'PSBC01411', 'currency_code': '$'},
    {'start_date': '2024-01-01', 'end_date': '2025-12-31', 'project_manager_name': 'Noah Benbazza',
     'project_id': 'PSBC02100', 'total_cost': '100,200.00', 'project_no': '1', 'currency_code': 'USD'},
    {'start_date': 'Jan 1st, 2025', 'end_date': 'Dec 31st, 2026', 'project_manager_name': 'S. Jacob Nesa Raj',
     'project_id': 'PSBC02138', 'total_cost': '$434,202', 'project_no': 'PSBC02138', 'currency_code': '$'},
    {'start_date': '01st February 2025', 'end_date': 'January 31, 2026', 'project_manager_name': 'Joseph DeLuca',
     'project_id': 'PSBC02188', 'total_cost': '$44,040', 'project_no': 'GEHC Payment Tracking and S2P Support 2025-26', 'currency_code': '$'},
    {'start_date': 'Mar 01, 2025', 'end_date': 'Feb 28, 2026', 'project_manager_name': 'Scott Velasquez',
     'project_id': 'PSBC02203', 'total_cost': '$70,000', 'project_no': '6', 'currency_code': '$'},
    {'start_date': '3rd-February-2025', 'end_date': '31st-January-2026', 'project_manager_name': 'Ramneet Saluja',
     'project_id': 'GEHC CRM Integration Interface Support', 'total_cost': '145,152', 'project_no': None, 'currency_code': 'USD'}
]

# DB Connection
connection = psycopg2.connect(
    host="localhost",
    port="5432",
    database="postgres",
    user="postgres",
    password="password"
)
cursor = connection.cursor()

# Create updated table
create_table_query = """
CREATE TABLE IF NOT EXISTS Coupa_contracts (
    project_manager_name VARCHAR(100),
    project_id VARCHAR(100) PRIMARY KEY,
    project_no VARCHAR(50),
    start_date DATE,
    end_date DATE,
    total_cost NUMERIC,
    currency_code VARCHAR(10)
);
"""
cursor.execute(create_table_query)
connection.commit()

# Insert query
insert_query = """
INSERT INTO Coupa_contracts 
(project_manager_name, project_id, project_no, start_date, end_date, total_cost, currency_code)
VALUES (%s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (project_id) DO NOTHING;
"""

# Insert data
for record in data:
    try:
        start_date = parser.parse(record["start_date"]).date()
        end_date = parser.parse(record["end_date"]).date()
    except Exception as e:
        print(f"❌ Date parse failed for {record['project_id']}: {e}")
        continue

    # Clean cost: remove $ and , and convert to float
    cost_str = str(record.get("total_cost", "0")).replace("$", "").replace(",", "").strip()
    try:
        total_cost = float(cost_str)
    except ValueError:
        total_cost = None

    values = (
        record["project_manager_name"],
        record["project_id"],
        record.get("project_no"),
        start_date,
        end_date,
        total_cost,
        record.get("currency_code")
    )
    cursor.execute(insert_query, values)

connection.commit()

# Fetch and print inserted data
cursor.execute("SELECT * FROM Coupa_contracts;")
for row in cursor.fetchall():
    print(row)

cursor.close()
connection.close()


('Sabin Dcruz', 'GEHC OTR Analytics 2024', 'CO#1', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('42000.0'), '$')
('Rama Singh', 'PSBC02046', '8', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('98112.0'), 'USD')
('Rama Singh', 'PSBC02045', '8', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('147168.0'), 'USD')
('Rama Singh', 'PSBC02062', '3', datetime.date(2025, 2, 1), datetime.date(2026, 1, 31), Decimal('128400.0'), 'USD')
('Rama Singh', 'PSBC02290', '3', datetime.date(2025, 2, 1), datetime.date(2026, 1, 31), Decimal('36000.0'), 'USD')
('Lisa Prestegaard', 'PSBC02078', 'PSBC01411', datetime.date(2023, 10, 1), datetime.date(2025, 12, 31), Decimal('788256.0'), '$')
('Noah Benbazza', 'PSBC02100', '1', datetime.date(2024, 1, 1), datetime.date(2025, 12, 31), Decimal('100200.0'), 'USD')
('S. Jacob Nesa Raj', 'PSBC02138', 'PSBC02138', datetime.date(2025, 1, 1), datetime.date(2026, 12, 31), Decimal('434202.0'), '$')
('Joseph DeLuca', 'PSBC0

In [13]:
# connection.rollback()  # <-- important
# cursor.execute("DROP TABLE IF EXISTS Coupa_contracts;")
# connection.commit()

# cursor.execute("""
# CREATE TABLE Coupa_contracts (
#     project_manager_name VARCHAR(100),
#     project_id VARCHAR(100) PRIMARY KEY,
#     project_no VARCHAR(50),
#     start_date DATE,
#     end_date DATE,
#     total_cost NUMERIC,
#     currency_code VARCHAR(10)
# );
# """)
# connection.commit()


In [14]:
# Fetch records
# DB Connection
import psycopg2
from dateutil import parser
connection = psycopg2.connect(
    host="localhost",
    port="5432",
    database="postgres",
    user="postgres",
    password="password"
)
cursor = connection.cursor()
cursor.execute("SELECT * FROM coupa_contracts;")
rows = cursor.fetchall()
for row in rows:
    print(row)

#Close connection
cursor.close()
connection.close()

('Sabin Dcruz', 'GEHC OTR Analytics 2024', 'CO#1', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('42000.0'), '$')
('Rama Singh', 'PSBC02046', '8', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('98112.0'), 'USD')
('Rama Singh', 'PSBC02045', '8', datetime.date(2025, 1, 1), datetime.date(2025, 12, 31), Decimal('147168.0'), 'USD')
('Rama Singh', 'PSBC02062', '3', datetime.date(2025, 2, 1), datetime.date(2026, 1, 31), Decimal('128400.0'), 'USD')
('Rama Singh', 'PSBC02290', '3', datetime.date(2025, 2, 1), datetime.date(2026, 1, 31), Decimal('36000.0'), 'USD')
('Lisa Prestegaard', 'PSBC02078', 'PSBC01411', datetime.date(2023, 10, 1), datetime.date(2025, 12, 31), Decimal('788256.0'), '$')
('Noah Benbazza', 'PSBC02100', '1', datetime.date(2024, 1, 1), datetime.date(2025, 12, 31), Decimal('100200.0'), 'USD')
('S. Jacob Nesa Raj', 'PSBC02138', 'PSBC02138', datetime.date(2025, 1, 1), datetime.date(2026, 12, 31), Decimal('434202.0'), '$')
('Joseph DeLuca', 'PSBC0

In [15]:
# Rename Coupa_contracts.project_no -> "Original_SOW_ref"
import psycopg2
from dateutil import parser
connection = psycopg2.connect(
    host="localhost",
    port="5432",
    database="postgres",
    user="postgres",
    password="password"
)
cursor = connection.cursor()
# cursor.execute('ALTER TABLE "coupa_contracts" RENAME COLUMN original_sow_ref TO "project_no";')
# connection.commit()


In [16]:
# cursor.execute("""SELECT project_id
# FROM coupa_contracts
# WHERE total_cost > 100 AND currency_code = 'USD';""")
# connection.commit()
cursor.execute(
    "SELECT project_id FROM public.coupa_contracts WHERE total_cost > 10 AND currency_code = 'USD';",
    (100, 'USD')
)
for (project_id,) in cursor.fetchall():
    print(project_id)


PSBC02046
PSBC02045
PSBC02062
PSBC02290
PSBC02100
GEHC CRM Integration Interface Support


In [7]:
cursor.execute("DELETE FROM public.coupa_contracts;")
connection.commit()
